## 02 — 검색 평가 (Search Evaluation)

- **입력:** `ground_truth-new.csv` (학생 질문 + 정답 document id)
- **검색:** minsearch 인덱스 (FAQ question / section / answer)
- **지표:** Hit Rate, MRR
- **튜닝:** boost 가중치 그리드 서치


In [9]:
import pandas as pd  # pd.read_csv, DataFrame 처리를 위해 pandas를 import한다.
from evaluation_paths import GROUND_TRUTH_CSV  # GROUND_TRUTH_CSV를 import해서 노트북 cwd와 무관하게 같은 CSV 경로를 쓴다.
                                               # 설정 파일에서 CSV 파일의 경로 문자열을 가져옴

# 경로를 직접 입력하지 않고 변수를 사용해 코드 실행 위치가 어디든 동일한 파일을 참조하도록 고정함
df_ground_truth = pd.read_csv(GROUND_TRUTH_CSV)  # read_csv()로 01 단계에서 만든 평가용 질문 CSV를 메모리에 올린다.

In [10]:
# 데이터프레임을 리스트로 변환
df_ground_truth.head()  # head()로 컬럼(question, course, document)과 값 형태를 먼저 눈으로 확인한다.
# 데이터프레임의 각 행을 딕셔너리 리스트 형태로 변환하여 루프 처리 준비

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf


In [11]:
ground_truth = df_ground_truth.to_dict(orient="records")  # to_dict()를 다시 호출해 ground_truth 변수명으로 평가 루프에 넘길 리스트를 확정한다.

In [12]:
ground_truth[10]  # ground_truth[10]으로 샘플 한 건의 question/document 키 구조를 확인한다.

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [13]:
from ingest import load_faq_data, build_index  # load_faq_data, build_index를 import해서 FAQ 원문과 검색 인덱스를 준비한다.

documents = load_faq_data()  # load_faq_data()로 검색 대상 FAQ 문서 전체를 불러온다.

documents_llm = []  # documents_llm = []로 llm-zoomcamp만 담을 빈 리스트를 만든다.

for doc in documents:  # for doc in documents로 전체 FAQ를 순회한다.
    if doc["course"] == "llm-zoomcamp":  # doc["course"] == "llm-zoomcamp"이면 이 과정 FAQ만 남긴다.
        documents_llm.append(doc)  # append(doc)로 필터된 문서를 리스트에 넣는다.

documents = documents_llm  # documents = documents_llm으로 검색·평가 대상 문서 집합을 고정한다.
index = build_index(documents)  # build_index(documents)로 question/section/answer 필드 기반 minsearch 인덱스를 만든다.

In [14]:
# 검색 엔진 가중치(boost) 설정 및 스모크 테스트
# 검색 알고리즘이 올바르게 동작하는지 샘플 질문으로 상위 결과를 미리 확인합니다.

boost = {"question": 3.0}  # boost dict로 question 필드 검색 가중치를 3.0으로 설정한다.

index.search(  # index.search()로 인덱스가 실제로 동작하는지 스모크 테스트한다.
    "What is the course about?",  # 첫 인자 query에 테스트 질문 문자열을 넣는다.
    num_results=5,  # num_results=5로 상위 5개만 받아 이후 top-5 평가 설정과 맞춘다.
    boost_dict=boost,  # boost_dict=boost로 question 가중치가 반영되는지 확인한다.
)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTu

In [15]:
def text_search(query):  # text_search(query) 함수를 정의해 이후 evaluate()에 search_function으로 넘긴다.
    boost_dict = {"question": 3.0, "section": 0.5}  # boost_dict로 question·section 필드별 가중치를 정한다.

    return index.search(  # index.search()를 return해서 호출부에서 검색 결과 리스트를 받게 한다.
        query,  # query 인자로 학생 질문 문자열을 그대로 검색어로 쓴다.
        num_results=5,  # num_results=5로 Hit Rate/MRR 계산에 쓰는 top-5를 유지한다.
        boost_dict=boost_dict,  # boost_dict로 필드별 점수 비중을 검색에 반영한다.
    )

In [16]:
q = ground_truth[0]  # 첫 번째 질문 레코드 선택
                     # ground_truth[0]에서 평가 파이프라인 테스트용 샘플 질문 하나를 꺼낸다.
q # q를 출력해 question, document 키가 있는지 확인한다.

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [17]:
doc_id = q["document"]  # q["document"]로 이 질문의 정답 FAQ id를 저장한다.
                        # 정답 문서 ID 추출
doc_id  # doc_id를 출력해 이후 검색 결과 id와 비교할 기준값을 확인한다.
        # 해당 질문으로 검색 수행

'74eb249bbf'

In [18]:
results = text_search(q["question"])  # 해당 질문으로 검색 수행
# text_search()에 q["question"]을 넣어 실제 검색 top-5를 받는다.

# 검색된 각 문서의 ID가 정답 문서 ID와 일치하는지 출력하여 확인
results  # results를 출력해 어떤 FAQ id들이 나오는지 본다.

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is a solo project.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '86d99bbf21',
  'course': 'llm-zoomc

In [19]:
# 검색된 각 문서의 ID가 정답 문서 ID와 일치하는지 출력하여 확인
for d in results: # for d in results로 top-5 각 문서를 순회한다.
                  # results를 출력해 어떤 FAQ id들이 나오는지 본다.
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')  # d["id"] == doc_id를 출력해 순위별로 정답 여부를 눈으로 확인한다.

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
86d99bbf21 == 74eb249bbf: False
489dd1c9d9 == 74eb249bbf: False


In [20]:
relevance = []  # relevance = []로 순위별 0/1 벡터를 담을 리스트를 만든다.

for d in results:  # for d in results로 검색 결과 각 순위를 본다.
    relevance.append(int(d["id"] == doc_id))  # append(int(...))로 정답 id면 1, 아니면 0을 넣어 MRR 입력을 만든다.
                                              # 일치하면 1, 아니면 0을 리스트에 추가

relevance  # relevance를 출력해 [1,0,0,0,0] 같은 형태를 확인한다.
           # 결과 예시: [1, 0, 0, 0, 0] (첫 번째 결과가 정답인 경우)

[1, 0, 0, 0, 0]

In [21]:
def compute_relevance_text(q):  # compute_relevance_text(q)로 질문 1건의 relevance 벡터를 계산하는 함수를 정의한다.
    doc_id = q["document"]  # q["document"]로 정답 id를 꺼내 검색 결과와 비교한다.
    results = text_search(query=q["question"])  # text_search(query=...)로 이 질문에 대한 top-5 검색을 수행한다.

    relevance = []  # relevance = []로 이 질문 전용 0/1 리스트를 초기화한다.
    for d in results:  # for d in results로 top-5를 순회한다.
        relevance.append(int(d["id"] == doc_id))  # append(int(...))로 순위별 hit 여부를 기록한다.

    return relevance  # return relevance로 한 질문의 평가 벡터를 호출부에 돌려준다.

In [22]:
# 첫 번째 ground truth 레코드를 사용하여 관련성 리스트 확인
q = ground_truth[0]  # ground_truth[0]으로 첫 질문 샘플을 고른다.
print(q["question"])  # print(q["question"])로 어떤 질문인지 확인한다.
compute_relevance_text(q)  # compute_relevance_text(q)를 호출해 보통 [1,0,0,0,0] 같은 케이스를 본다.
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [23]:
q = ground_truth[11]  # ground_truth[11]로 정답이 3위에 오는 다른 샘플을 고른다.
print(q["question"])  # print()로 질문 내용을 확인한다.
compute_relevance_text(q)  # compute_relevance_text()로 순위가 낮을 때 relevance 형태를 본다.
# [0, 0, 1, 0, 0]

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

In [24]:
[0, 0, 1, 0, 0]  # 위 셀 결과 참고: 정답이 3위일 때 relevance 벡터 예시.

[0, 0, 1, 0, 0]

In [25]:
q = ground_truth[50]  # ground_truth[50]으로 검색 실패(전부 0) 케이스를 본다.
print(q["question"])  # print()로 질문을 확인한다.
compute_relevance_text(q)  # compute_relevance_text()로 miss 케이스 relevance를 확인한다.
# [0, 0, 0, 0, 0]

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [21]:
[0, 0, 0, 0, 0]  # 위 셀 결과 참고: 정답 문서가 top-5에 없을 때 벡터 예시.

[0, 0, 0, 0, 0]

In [27]:
from tqdm.auto import tqdm  # tqdm을 import해서 전체 ground_truth 루프 진행률을 표시한다.


def compute_relevance_total_text(ground_truth):  # compute_relevance_total_text()로 전체 질문 relevance 리스트를 만든다.
    relevance_total = []  # relevance_total = []로 질문별 relevance 벡터들을 모을 리스트를 초기화한다.

    for q in tqdm(ground_truth):  # for q in tqdm(ground_truth)로 395건 전체를 순회한다.
        relevance = compute_relevance_text(q)  # compute_relevance_text(q)로 질문 하나의 0/1 벡터를 구한다.
        relevance_total.append(relevance)  # append(relevance)로 전체 결과 리스트에 쌓는다.

    return relevance_total  # return relevance_total로 Hit Rate/MRR 입력 데이터를 돌려준다.

In [29]:
# 먼저 전체를 실행하기 전에, 지침에 따라 처음 15개의 질문으로 샘플을 만들어 테스트해 봅니다.
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

# 2026년 5월 29일 기준으로 준비된 데이터의 경우, 출력 결과는 다음과 같습니다:
relevance_total_text

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [31]:
relevance = compute_relevance_total_text(ground_truth)  # compute_relevance_total_text()를 실행해 전체 relevance를 계산한다.

  0%|          | 0/395 [00:00<?, ?it/s]

In [32]:
relevance[:15]  # relevance[:15]로 앞 15건만 잘라 중간 결과 형태를 빠르게 확인한다.

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [33]:
def compute_relevance(q, search_function):  # compute_relevance()에 search_function 인자를 넣어 boost 튜닝 시 함수만 바꿀 수 있게 한다.
    doc_id = q["document"]  # q["document"]로 정답 id를 가져온다.
    results = search_function(query=q["question"])  # search_function(query=...)로 text_search 대신 다른 검색 함수도 쓸 수 있게 한다.

    relevance = []  # relevance = []로 0/1 벡터를 초기화한다.
    for d in results:  # for d in results로 top-5를 순회한다.
        relevance.append(int(d["id"] == doc_id))  # append(int(...))로 hit 여부를 기록한다.

    return relevance  # return relevance로 한 질문 평가 결과를 반환한다.

In [34]:
def compute_relevance_total(ground_truth, search_function):  # compute_relevance_total()로 전체 질문 + 임의 검색 함수 조합을 평가한다.
    relevance_total = []  # relevance_total = []로 전체 결과를 담을 리스트를 만든다.

    for q in tqdm(ground_truth):  # for q in tqdm(ground_truth)로 모든 질문을 돈다.
        relevance = compute_relevance(q, search_function)  # compute_relevance(q, search_function)로 검색 함수별 relevance를 구한다.
        relevance_total.append(relevance)  # append()로 질문별 벡터를 모은다.

    return relevance_total  # return relevance_total로 evaluate()에 넘길 데이터를 만든다.

In [35]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total(ground_truth_sample, text_search)
relevance_total_text

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [36]:
relevance_total = compute_relevance_total(ground_truth, text_search)  # compute_relevance_total(..., text_search)로 기본 검색 설정의 전체 relevance를 계산한다.

  0%|          | 0/395 [00:00<?, ?it/s]

In [38]:
sample = relevance_total[:15]  # relevance_total[:15]로 15건만 잘라 지표 수동 계산·이해용 샘플을 만든다.

In [29]:
sample  # sample을 출력해 15개 relevance 벡터를 확인한다.

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [30]:
14 / 15  # 14/15로 샘플 15건 중 14건 hit라는 수동 계산 예시를 보여준다.

0.9333333333333333

In [39]:
# cnt를 0으로 초기화하여 정답이 포함된 질문의 개수를 셀 준비를 합니다.
cnt = 0  

# sample에 담긴 질문별 relevance 벡터를 하나씩 순회합니다.
for line in sample:  
    # 만약 정답 문서가 상위 5위 안에 있다면(1이 포함되어 있다면), hit으로 간주합니다.
    if 1 in line:  
        # 정답을 찾은 질문의 개수를 1 증가시킵니다.
        cnt = cnt + 1  

# 전체 샘플 개수(len(sample))로 적중 횟수(cnt)를 나누어 Hit Rate를 계산합니다.
cnt / len(sample)  

0.9333333333333333

In [40]:
def hit_rate(relevance):  # hit_rate(relevance) 함수로 전체 relevance에서 Hit Rate를 계산한다.
    cnt = 0  # cnt = 0으로 hit 카운터를 초기화한다.
             # 정답이 포함된 질문의 개수를 담을 변수 초기화

    for line in relevance:  # for line in relevance로 모든 질문의 relevance 벡터를 본다.
                            # 각 질문의 관련성 리스트(0, 1)를 순회
        if 1 in line:  # if 1 in line이면 해당 질문은 검색 성공(hit)이다.
                       # 정답 문서가 검색 결과에 포함되어 있다면(1이 존재하면)
            cnt = cnt + 1  # 적중 횟수 증가. cnt를 올려 hit 개수를 센다.

    return cnt / len(relevance)  # return cnt/len(relevance)로 hit 비율을 반환한다.
                                # 전체 질문 개수로 나누어 비율 반환

In [41]:
# 예시 데이터로 확인 (앞서 만든 sample 사용)
# hit_rate(relevance)를 호출해 text_search 기준 전체 Hit Rate를 본다.
hit_rate(sample)

0.9333333333333333

In [42]:
total_score = 0.0  # total_score = 0.0으로 MRR 합계를 초기화한다.


for line in sample:  # for line in sample으로 샘플 질문별 relevance를 순회한다.
    for rank in range(len(line)):  # for rank in range(len(line))으로 0,1,2,... 순위를 본다.
        if line[rank] == 1:  # line[rank] == 1이면 그 순위에 정답 문서가 있다.
            score = 1 / (rank + 1)  # score = 1/(rank+1)로 reciprocal rank 점수를 구한다.
            total_score = total_score + score  # total_score에 더해 질문당 첫 정답 순위 점수만 반영한다.
            break  # break로 한 질문에서 첫 번째 정답 순위만 MRR에 넣는다.

total_score / len(sample)  # total_score/len(sample)로 샘플 MRR을 수동 계산한다.

0.8222222222222222

In [43]:
def mrr(relevance):  # mrr(relevance)로 Mean Reciprocal Rank를 계산하는 함수를 정의한다.
                     # 각 질문별 점수를 합산할 변수 초기화
    total_score = 0.0  # total_score = 0.0으로 역순위 점수 합을 초기화한다.

    for line in relevance:  # for line in relevance로 각 질문의 relevance 벡터를 본다.
                            # 각 질문의 관련성 리스트 순회
        for rank in range(len(line)):  # for rank로 top-5 순위를 순회한다. 검색 결과의 순위(rank)를 확인
            if line[rank] == 1:  # line[rank] == 1이면 그 순위가 정답 문서 위치이다. 정답 문서가 해당 순위에서 발견되면
                score = 1 / (rank + 1)  # 순위에 따른 역수 점수 합산 score = 1/(rank+1)로 reciprocal rank를 계산한다.
                total_score = total_score + score  # total_score에 더한다.
                break  # break로 질문당 첫 정답만 반영한다. 정답을 찾았으므로 다음 질문으로 이동. 

    return total_score / len(relevance)  # 전체 평균 점수 반환. return total_score/len(relevance)로 평균 MRR을 반환한다.

In [44]:
mrr(sample)  # mrr(sample)로 15건 샘플 MRR을 확인한다.

0.8222222222222222

In [45]:
mrr(relevance)  # mrr(relevance)로 전체 395건 MRR을 확인한다.

0.7693248945147676

In [46]:
# 통합 평가 함수: 검색 함수를 인자로 받아 Hit Rate와 MRR을 한 번에 계산
def evaluate(ground_truth, search_function):  # evaluate()로 relevance 계산 + Hit Rate + MRR을 한 번에 수행한다. 전체 데이터에 대한 관련성 리스트를 먼저 계산
    relevance_total = compute_relevance_total(ground_truth, search_function)  # compute_relevance_total()로 해당 검색 함수의 전체 relevance를 만든다.
    
    # 지표를 딕셔너리 형태로 반환
    return {  # return dict로 두 지표를 함께 돌려준다.
        "hit_rate": hit_rate(relevance_total),  # hit_rate()로 top-5에 정답이 포함된 비율을 계산한다.
        "mrr": mrr(relevance_total),  # mrr()로 정답 순위의 역수 평균을 계산한다.
    }

In [47]:
# 전체 데이터셋(ground_truth)과 검색 함수(text_search)를 사용하여 최종 평가 수행
evaluate(ground_truth, text_search)  # evaluate(..., text_search)로 기본 boost 설정의 성능을 본다.

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.8987341772151899, 'mrr': 0.7693248945147676}

In [48]:
def text_search_v2(query):  # text_search_v2()로 question boost를 낮춘 변형 검색을 정의한다.
    boost_dict = {"question": 2.0, "section": 0.5}  # boost_dict에서 question=2.0으로 v1(3.0)과 비교한다.

    return index.search(  # index.search()로 동일 인덱스에 다른 가중치를 적용한다.
        query,  # query로 학생 질문을 검색한다.
        num_results=5,  # num_results=5로 평가 설정을 유지한다.
        boost_dict=boost_dict,  # boost_dict로 v2 가중치를 반영한다.
    )

In [49]:
evaluate(ground_truth, text_search_v2)  # evaluate(..., text_search_v2)로 boost 변경 효과를 숫자로 비교한다.

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.9088607594936708, 'mrr': 0.791561181434599}

In [50]:
# question 필드에 대한 가중치(boost)를 조절할 수 있는 검색 함수
def search_boost(query, question_boost):  # search_boost()로 question boost만 바꿔가며 실험한다.
    # question에는 파라미터로 받은 가중치를, section에는 0.5를 고정으로 부여
    boost_dict = {"question": question_boost, "section": 0.5}  # boost_dict에 question_boost 인자를 넣는다.

    return index.search(  # index.search()로 각 boost 값마다 top-5를 받는다.
        query,  # query에 질문 문자열을 넣는다.
        num_results=5,  # num_results=5로 고정한다.
        boost_dict=boost_dict,  # boost_dict로 가중치를 적용한다.
    )

In [51]:
# 테스트할 가중치 목록
# evaluate()로 검색 성능을 평가할 때, lambda를 사용하여 반복문의 boost 값을 고정합니다.
# lambda query, boost=boost: search_boost(query, boost)
# 1. query: evaluate()가 내부적으로 ground_truth의 질문(q["question"])을 매개변수로 전달합니다.
# 2. boost=boost: 현재 반복문의 boost 값을 lambda 내부의 기본 매개변수로 고정(capture)하여 search_boost에 전달합니다.


for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:  # for boost in [...]로 question 가중치 후보를 순회한다.
    result = evaluate(  # evaluate()로 각 boost마다 Hit Rate/MRR을 계산한다.
        ground_truth,  # ground_truth로 동일 평가 집합을 쓴다.
        lambda query, boost=boost: search_boost(query, boost),  # lambda로 boost 값을 search_boost에 고정해 넘긴다.
    )
    print(f"boost={boost}: {result}")  # print()로 boost별 결과를 바로 비교한다.

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9113924050632911, 'mrr': 0.800548523206751}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9240506329113924, 'mrr': 0.8139240506329113}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8987341772151899, 'mrr': 0.7693248945147676}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8708860759493671, 'mrr': 0.7401265822784809}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8582278481012658, 'mrr': 0.7122362869198313}


In [53]:
def search_boosts(query, question_boost, answer_boost, section_boost):  # search_boosts()로 세 필드 boost를 동시에 튜닝한다.
    boost_dict = {  # boost_dict에 question/section/answer 가중치를 모두 넣는다.
        "question": question_boost,  # question_boost로 질문 필드 비중을 조절한다.
        "section": section_boost,  # section_boost로 섹션 필드 비중을 조절한다.
        "answer": answer_boost,  # answer_boost로 답변 필드 비중을 조절한다.
    }

    return index.search(  # index.search()로 조합별 top-5를 반환한다.
        query,  # query로 검색한다.
        num_results=5,  # num_results=5로 유지한다.
        boost_dict=boost_dict,  # boost_dict로 세 가중치를 반영한다.
    )

In [ ]:
results = []  # results = []로 그리드 서치 결과를 담을 리스트를 만든다.

for question_boost in [1.0, 2.0, 5.0]:  # for question_boost로 question 가중치 후보를 순회한다.
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:  # for answer_boost로 answer 가중치 후보를 순회한다.
        for section_boost in [0.1, 0.2, 0.5]:  # for section_boost로 section 가중치 후보를 순회한다.
            print(  # print()로 현재 조합을 로그에 남긴다.
                f"Evaluating question_boost={question_boost}, "
                f"answer_boost={answer_boost}, section_boost={section_boost}..."
            )
            result = evaluate(  # evaluate()로 이 boost 조합의 Hit Rate/MRR을 계산한다.
                ground_truth,  # ground_truth 전체로 평가한다.
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,  # query를 search_boosts에 전달한다.
                    question_boost,  # question_boost를 고정한다.
                    answer_boost,  # answer_boost를 고정한다.
                    section_boost,  # section_boost를 고정한다.
                ),
            )

            results.append({  # results.append()로 조합별 지표를 저장한다.
                "question": question_boost,  # question 키에 question_boost 값을 기록한다.
                "answer": answer_boost,  # answer 키에 answer_boost 값을 기록한다.
                "section": section_boost,  # section 키에 section_boost 값을 기록한다.
                "hit_rate": result["hit_rate"],  # hit_rate를 저장해 나중에 정렬한다.
                "mrr": result["mrr"],  # mrr을 저장해 최적 조합을 고른다.
            })

In [55]:
df_results = pd.DataFrame(results)  # DataFrame(results)로 그리드 서치 결과를 표로 만든다.
df_results.sort_values("mrr", ascending=False).head(10)  # sort_values("mrr")로 MRR 상위 10개 boost 조합을 본다.

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.974684,0.884515
19,2.0,4.0,0.2,0.974684,0.884515
35,5.0,10.0,0.5,0.974684,0.884515
34,5.0,10.0,0.2,0.974684,0.883797
33,5.0,10.0,0.1,0.974684,0.883671
18,2.0,4.0,0.1,0.974684,0.883671
20,2.0,4.0,0.5,0.977215,0.883629
4,1.0,2.0,0.2,0.977215,0.883544
5,1.0,2.0,0.5,0.964557,0.862447
6,1.0,4.0,0.1,0.969620,0.861561


In [57]:
# 최종 시스템 확정: 데이터가 가리키는 최적의 비율을 사용하여 
# 향후 모든 검색에 사용할 표준 text_search 함수를 정의하십시오.

# 예시: 최적 비율(1.0, 2.0, 0.1) 적용
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }
    return index.search(query, num_results=5, boost_dict=boost_dict)

# 이 과정을 통해 직관이 아닌 데이터(Evidence)에 근거하여 검색 시스템을 확정하게 됩니다.
# 분석하신 상위 10개 결과 중 어떤 비율이 가장 높게 나왔는지 확인해 보시기 바랍니다.

In [58]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [ ]:
relevance_total_text

In [60]:
# 상위 10개 결과 출력하여 최적의 부스트 값 조합 확인
best_combination = df_results.sort_values("mrr", ascending=False).iloc[0]
print("최적의 부스트 값 조합:")
print(best_combination)

최적의 부스트 값 조합:
question    1.000000
answer      2.000000
section     0.100000
hit_rate    0.974684
mrr         0.884515
Name: 3, dtype: float64


In [61]:
# 파일명: search_tuning.ipynb (혹은 해당 노트북)

def text_search(query):
    # 위에서 찾은 최적의 값을 여기에 입력합니다.
    boost_dict = {
        "question": float(best_combination["question"]),
        "answer": float(best_combination["answer"]),
        "section": float(best_combination["section"]),
    }
    return index.search(query, num_results=5, boost_dict=boost_dict)

In [64]:
# 결과 중 첫 번째 항목만 출력하여 구조 파악
sample_hit = results[0]
print("결과 딕셔너리의 키 값들:", sample_hit.keys())

결과 딕셔너리의 키 값들: dict_keys(['id', 'course', 'section', 'question', 'answer'])


In [65]:
# 'score' 키를 제외하고 실제 존재하는 키들을 사용하여 결과 출력
for i, hit in enumerate(results):
    print(f"{i+1}: {hit['question']}")
    print(f"   Answer: {hit['answer'][:100]}...") # 답변 내용이 길 수 있으니 100자 정도만 출력
    print("-" * 30)

1: Run MCP Inspector
   Answer: To run the MCP Inspector, execute the following command in the terminal:

```bash
npx @modelcontextp...
------------------------------
2: OpenAI: Error: RateLimitError: Error code: 429 -
   Answer: ```json
RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, pl...
------------------------------
3: How to Solve "RuntimeError: Already running asyncio in this thread"
   Answer: Jupyter notebooks already run an event loop in the main thread to handle asynchronous code. For this...
------------------------------
4: Agents: "RuntimeError: Already running asyncio in this thread" when calling asyncio.run() from Jupyter
   Answer: Jupyter already runs an event loop inside the kernel, so calling `asyncio.run(...)` blows up with:

...
------------------------------
5: How do I know which tables are in the db?
   Answer: You can use the `db.table_names()` method to list all the tables in the database....
-------------------------

In [66]:
# 전체 필드를 다 보고 싶을 때
for i, hit in enumerate(results):
    print(f"{i+1}: {hit['question']}")
    print(f"   Course: {hit['course']} | Section: {hit['section']}")
    print(f"   Answer: {hit['answer'][:100]}...")
    print("-" * 30)

1: Run MCP Inspector
   Course: llm-zoomcamp | Section: Module 2: Agents
   Answer: To run the MCP Inspector, execute the following command in the terminal:

```bash
npx @modelcontextp...
------------------------------
2: OpenAI: Error: RateLimitError: Error code: 429 -
   Course: llm-zoomcamp | Section: Module 1: Introduction to LLMs and RAG
   Answer: ```json
RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, pl...
------------------------------
3: How to Solve "RuntimeError: Already running asyncio in this thread"
   Course: llm-zoomcamp | Section: Module 2: Agents
   Answer: Jupyter notebooks already run an event loop in the main thread to handle asynchronous code. For this...
------------------------------
4: Agents: "RuntimeError: Already running asyncio in this thread" when calling asyncio.run() from Jupyter
   Course: llm-zoomcamp | Section: Module 2: Agents
   Answer: Jupyter already runs an event loop inside the kernel, so calling `async